<img src="https://cros.ec.europa.eu/system/files/inline-images/logo%20official%20estp_0.jpg" height="120px">

# ADVANCED PYTHON FOR OFFICIAL STATISTICS
## ICON-Institute · Cologne · June 22–26, 2026 · Day 3 AM
### [Dr. Christian Kauth](https://www.linkedin.com/in/ckauth/)

---

# 🔢 Notebook 05 — NumPy, statsmodels & ML Pipelines
## *Advanced Python for Official Statistics*

> *"All models are wrong, but some are useful."* — George Box

This morning we go numerical. We replace our pure-Python Gini with a vectorised NumPy version,
fit regression models with `statsmodels`, and build a scikit-learn pipeline that classifies
households as at-risk-of-poverty — then explain every prediction with SHAP.

| Topic | You will be able to… |
|-------|---------------------|
| **NumPy vectorisation** | Replace loops with array ops; 100× faster Gini |
| **Weighted statistics** | Design-weighted AROP and Gini (correct for complex survey design) |
| **statsmodels OLS** | Regress equivalised income on education, age, household type |
| **scikit-learn pipeline** | Impute → encode → scale → classify with one `.fit()` call |
| **Cross-validation** | Estimate generalisation error without data leakage |
| **SHAP** | Explain individual poverty-risk predictions to policymakers |

In [ ]:
import os
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.toml not found")

PROJECT_ROOT = find_project_root(Path().resolve())
os.chdir(PROJECT_ROOT)

DATA_DIR    = PROJECT_ROOT / "data"
PARQUET_DIR = DATA_DIR / "parquet"
print(f"✅ Project root: {PROJECT_ROOT}")


✅ Project root: D:\Local\ICON\Python_Advanced\python_advanced


### `pyproject.toml` update — Notebook 05

This notebook adds: **numpy>=1.26, scipy>=1.11, statsmodels>=0.14, scikit-learn>=1.4, shap>=0.44, matplotlib>=3.8**

Run the cell below to update `pyproject.toml` and install the new packages.

In [ ]:
%%writefile pyproject.toml
# Day 5: adds NumPy, statsmodels, scikit-learn, SHAP
# New in notebook 05:
#   + numpy>=1.26
#   + scipy>=1.11
#   + statsmodels>=0.14
#   + scikit-learn>=1.4
#   + shap>=0.44
#   + matplotlib>=3.8

[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[project]
name = "silc-toolkit"
version = "0.1.0"
description = "Advanced Python for Official Statistics -- EU-SILC Analysis Toolkit"
requires-python = ">=3.11"
authors = [{ name = "Christian Kauth", email = "christian.kauth@unifr.ch" }]
license = { text = "MIT" }

dependencies = [
    "pydantic>=2.5",
    "pandas>=2.1",
    "pyarrow>=14.0",
    "polars>=0.20",
    "requests>=2.31",
    "beautifulsoup4>=4.12",
    "sqlalchemy>=2.0",
    "numpy>=1.26",
    "scipy>=1.11",
    "statsmodels>=0.14",
    "scikit-learn>=1.4",
    "shap>=0.44",
    "matplotlib>=3.8",
]

[project.optional-dependencies]
dev = [
    "pytest>=7.4",
    "pytest-cov>=4.1",
    "black>=23.0",
    "ruff>=0.3",
]

[project.urls]
Repository = "https://github.com/icon-institute/silc-toolkit"

[tool.hatch.build.targets.wheel]
packages = ["silc_toolkit"]

[tool.pytest.ini_options]
testpaths = ["tests"]
addopts = "-v --tb=short"

[tool.black]
line-length = 88
target-version = ["py311"]

[tool.ruff]
line-length = 88
target-version = "py311"

[tool.ruff.lint]
select = ["E", "F", "I", "UP"]
ignore = ["E501"]

[tool.ruff.lint.isort]
known-first-party = ["silc_toolkit"]

Overwriting pyproject.toml


In [ ]:
# Sync all dependencies declared in pyproject.toml
# Run this after every %%writefile pyproject.toml cell
import subprocess
result = subprocess.run(
    ["uv", "pip", "install", "-e", "."],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("silc-toolkit + notebook 05 deps installed")
else:
    print("uv error:", result.stderr[-500:])

silc-toolkit + notebook 05 deps installed


In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")   # non-interactive backend for nbconvert
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

print(f"   NumPy {np.__version__}  |  pandas {pd.__version__}")

   NumPy 2.4.6  |  pandas 3.0.3


---
## 1 · NumPy Vectorisation

### 1.1 The Problem with Pure Python

Our current Gini uses a Python loop over sorted incomes. For 100,000 households it's slow.
NumPy replaces loops with vectorised C operations — often 50–200× faster.

In [ ]:
import time
from silc_toolkit import load_incomes, gini_coefficient

def gini_numpy(incomes):
    # Vectorised Gini — sorted-rank formula: G = sum((2i-n-1)*x_i) / (n*sum(x_i))
    xs = np.sort(incomes[incomes > 0])
    n  = len(xs)
    if n == 0:
        return 0.0
    ranks   = np.arange(1, n + 1)
    weights = 2 * ranks - n - 1
    return float(np.dot(weights, xs) / (n * xs.sum()))


# Benchmark pure-Python vs NumPy
inc_lu = np.array(load_incomes("LU", 2012, DATA_DIR))

t0 = time.perf_counter()
for _ in range(1000):
    gini_coefficient(inc_lu.tolist())
t_py = (time.perf_counter() - t0) / 1000 * 1000

t0 = time.perf_counter()
for _ in range(1000):
    gini_numpy(inc_lu)
t_np = (time.perf_counter() - t0) / 1000 * 1000

print(f"Luxembourg 2012 Gini:")
print(f"  Pure Python : {gini_coefficient(inc_lu.tolist()):.6f}  ({t_py:.2f} ms)")
print(f"  NumPy       : {gini_numpy(inc_lu):.6f}  ({t_np:.2f} ms)")
print(f"  Speedup     : {t_py/t_np:.1f}x")

Luxembourg 2012 Gini:
  Pure Python : 0.374852  (2.23 ms)
  NumPy       : 0.374852  (0.07 ms)
  Speedup     : 33.9x


In [ ]:
def compute_equivalised_income_numpy(disposable_income, n_adults, n_children):
    # Modified OECD equivalence scale: 1 + 0.5*(adults-1) + 0.3*children
    scale = 1.0 + 0.5 * (n_adults - 1) + 0.3 * n_children
    return disposable_income / np.maximum(scale, 1.0)


# Demo on synthetic data
rng      = np.random.default_rng(42)
n_hh     = 5_000
income   = rng.lognormal(mean=10.5, sigma=0.8, size=n_hh)
n_adults = rng.integers(1, 5, size=n_hh)
n_child  = rng.integers(0, 4, size=n_hh)

equiv     = compute_equivalised_income_numpy(income, n_adults, n_child)
median_eq = float(np.median(equiv))
threshold = 0.6 * median_eq
arop_v    = float(np.mean(equiv < threshold))

print(f"Synthetic sample (n={n_hh:,}):")
print(f"  Median equiv income : EU{median_eq:,.0f}")
print(f"  AROP threshold      : EU{threshold:,.0f}")
print(f"  AROP rate           : {arop_v:.1%}")
print(f"  Gini (NumPy)        : {gini_numpy(equiv):.4f}")

Synthetic sample (n=5,000):
  Median equiv income : EU17,226
  AROP threshold      : EU10,335
  AROP rate           : 28.2%
  Gini (NumPy)        : 0.4594


---
## 2 · Regression with statsmodels

### 🧭 statsmodels vs scikit-learn — When to Use Which?

If you know already know scikit-learn: every model has `.fit()`, `.predict()`, `.score()` — beautiful uniform API. So why use statsmodels at all?

The key distinction is **inference vs prediction**:

| Question you're asking | Tool | What it gives you |
|------------------------|------|-------------------|
| *"Is education's effect on income statistically significant?"* | **statsmodels** | t-statistics, p-values, 95% CIs |
| *"How well does my model explain the variance in income?"* | **statsmodels** | R², adjusted R², F-test |
| *"Do my residuals violate OLS assumptions?"* | **statsmodels** | Durbin-Watson, Omnibus, Jarque-Bera |
| *"Can I predict whether this household is at risk?"* | **scikit-learn** | AUC, accuracy, recall |
| *"Which hyperparameters work best?"* | **scikit-learn** | `GridSearchCV`, `cross_val_score` |
| *"I need this to work for 20 different model types"* | **scikit-learn** | Uniform `.fit()/.predict()` API |

> **Rule of thumb:**  
> — Publishing regression coefficients in a policy report → **statsmodels**  
> — Building a classifier to flag at-risk households → **scikit-learn**  
>  
> In official statistics you often need *both*: statsmodels for the econometric report, scikit-learn for operational prediction.

---
### 2.1 Research Question

**Do education level and age predict household equivalised income?**

We link the P-file (persons 16+, with education and age) to the H-file (household income).

Variables:
- **Dependent:** `ln(equiv_income)` — log scale for right-skewed incomes
- **Independent:** age, age², ISCED education level (0–6), sex, country


In [ ]:
import csv

COUNTRIES_REG = ["BE", "ES", "FR", "IE", "LU", "SE"]
YEAR_REG      = 2012

person_records = []
for country in COUNTRIES_REG:
    p_path = DATA_DIR / f"{country}_PUF_EUSILC" / f"{country}_{YEAR_REG}p_EUSILC.csv"
    h_path = DATA_DIR / f"{country}_PUF_EUSILC" / f"{country}_{YEAR_REG}h_EUSILC.csv"
    if not p_path.exists() or not h_path.exists():
        continue

    # Build household-level lookup: income + structural housing features
    h_info = {}
    with open(h_path, encoding="utf-8") as fh:
        for row in csv.DictReader(fh):
            hid = int(row["HB030"])
            inc  = row.get("HY020", "")
            siz  = row.get("HX040", "1")
            ten  = row.get("HH021", "")   # tenure: 1=owner, 2=mortgage, 3=market rent, 4=reduced rent, 5=free
            warm = row.get("HH050", "")   # can keep home warm: 1=yes, 2=no
            try:
                income = float(inc) if inc and inc != "NA" else 0.0
                size   = max(1, int(float(siz))) if siz and siz != "NA" else 1
                h_info[hid] = {
                    "equiv":     income / (1.0 + 0.5 * (size - 1)),
                    "hh_size":   size,
                    "tenure":    int(float(ten))  if ten  and ten  not in ("NA", "") else None,
                    "warm_home": int(float(warm)) if warm and warm not in ("NA", "") else None,
                }
            except (ValueError, ZeroDivisionError):
                pass

    # Load persons — add employment status and marital status
    with open(p_path, encoding="utf-8") as fh:
        for row in csv.DictReader(fh):
            try:
                pid  = int(row["PB030"])
                hid  = pid // 10
                by   = int(row.get("PB140") or 0)
                age  = YEAR_REG - by if by > 1900 else None
                sex  = int(row.get("PB150") or 0)
                educ = row.get("PE040", "")
                educ = int(float(educ)) if educ and educ not in ("NA", "") else None
                # PL031: 1=FT employee, 2=PT employee, 3=FT self-emp, 4=PT self-emp,
                #         5=unemployed, 6=retired, 7=student, 8=disabled, 9=domestic, 10/11=other
                pl031 = row.get("PL031", "")
                empl  = int(float(pl031)) if pl031 and pl031 not in ("NA", "") else None
                pb190 = row.get("PB190", "")
                mart  = int(float(pb190)) if pb190 and pb190 not in ("NA", "") else None

                hh    = h_info.get(hid)
                equiv = hh["equiv"] if hh else None

                if age and equiv and equiv > 0 and educ is not None:
                    person_records.append({
                        "country":      country,
                        "age":          age,
                        "age_sq":       age ** 2,
                        "sex":          sex,
                        "education":    educ,
                        "equiv_income": equiv,
                        "ln_income":    np.log(equiv),
                        # ── new features ──────────────────────────────────
                        "employment":   empl,           # PL031
                        "marital":      mart,           # PB190
                        "hh_size":      hh["hh_size"]   if hh else None,   # HX040
                        "tenure":       hh["tenure"]    if hh else None,   # HH021
                        "warm_home":    hh["warm_home"] if hh else None,   # HH050
                    })
            except (ValueError, TypeError):
                pass

df_reg = pd.DataFrame(person_records)
print(f"Regression dataset: {len(df_reg):,} persons across {COUNTRIES_REG}")

# Show fill rates for the new features
new_cols = ["employment", "marital", "hh_size", "tenure", "warm_home"]
print("\nNew feature fill rates:")
for col in new_cols:
    fill = df_reg[col].notna().mean()
    print(f"  {col:<12}: {fill:.1%}")
print()
print(df_reg[["age", "education", "employment", "hh_size", "equiv_income"]].describe().round(1))


Regression dataset: 64,530 persons across ['BE', 'ES', 'FR', 'IE', 'LU', 'SE']

New feature fill rates:
  employment  : 99.7%
  marital     : 99.9%
  hh_size     : 100.0%
  tenure      : 100.0%
  warm_home   : 99.9%

           age  education  employment  hh_size  equiv_income
count  64530.0    64530.0     64363.0  64530.0       64530.0
mean      48.2        3.0         4.2      3.0       26630.4
std       17.9        1.5         3.1      1.4       25146.0
min       17.0        0.0         1.0      1.0           1.0
25%       34.0        2.0         1.0      2.0       11916.0
50%       47.0        3.0         4.0      3.0       20798.0
75%       62.0        5.0         7.0      4.0       34529.5
max       81.0        6.0        11.0     14.0     1048776.9


In [ ]:
import statsmodels.formula.api as smf

# Working-age persons with valid education and sex
df_ols = df_reg[
    (df_reg["age"].between(25, 64)) &
    (df_reg["education"].isin([1,2,3,4,5,6])) &
    (df_reg["sex"].isin([1,2]))
].copy()

df_ols["age_c"]    = df_ols["age"] - df_ols["age"].mean()
df_ols["age_sq_c"] = df_ols["age_c"] ** 2

formula = "ln_income ~ age_c + age_sq_c + C(education) + C(sex) + C(country)"
model   = smf.ols(formula, data=df_ols).fit()

print(model.summary2().tables[0])
print()
print("Key coefficients (age effect):")
print(model.params[["age_c", "age_sq_c"]].round(4))
print()

educ_cols = [c for c in model.params.index if "education" in c]
print("Education premium vs. ISCED 1 (low education):")
for col in educ_cols:
    level = col.split("[T.")[-1].rstrip("]")
    pct   = (np.exp(model.params[col]) - 1) * 100
    print(f"  ISCED {level}: {pct:+.1f}%")

                     0                 1                    2            3
0               Model:               OLS      Adj. R-squared:        0.140
1  Dependent Variable:         ln_income                 AIC:  114729.4105
2                Date:  2026-06-23 22:44                 BIC:  114850.9751
3    No. Observations:             43616      Log-Likelihood:      -57351.
4            Df Model:                13         F-statistic:        548.8
5        Df Residuals:             43602  Prob (F-statistic):         0.00
6           R-squared:             0.141               Scale:      0.81241

Key coefficients (age effect):
age_c       0.0089
age_sq_c    0.0003
dtype: float64

Education premium vs. ISCED 1 (low education):
  ISCED 2: +6.8%
  ISCED 3: +18.2%
  ISCED 4: +22.5%
  ISCED 5: +29.5%
  ISCED 6: +41.7%


In [ ]:
# ── Interpret OLS results ────────────────────────────────────────────────────
r2      = model.rsquared
adj_r2  = model.rsquared_adj
f_stat  = model.fvalue
n_obs   = int(model.nobs)
df_m    = int(model.df_model)
resid_s = model.scale ** 0.5          # residual std dev in log-income units

age_pct    = (np.exp(model.params["age_c"]) - 1) * 100
age_sq_val = model.params["age_sq_c"]
mean_age   = df_ols["age"].mean()
# vertex of the age parabola (in original age units)
vertex_age = mean_age + (-model.params["age_c"] / (2 * model.params["age_sq_c"])) \
             if model.params["age_sq_c"] != 0 else float("inf")

if age_sq_val > 0:
    # Positive quadratic → U-shape: vertex is the MINIMUM, not a peak
    age_sq_note = (
        f"  Positive quadratic → U-shaped profile; income trough is around age {vertex_age:.0f}.\n"
        f"  Workers aged 25–{int(vertex_age)} are in the (slight) declining part;\n"
        f"  from age {int(vertex_age)} to 64 income rises.  Because β₂ is near-zero,\n"
        "  the linear term dominates: the effective profile is nearly monotone."
    )
else:
    # Negative quadratic → inverted-U: vertex is the income PEAK
    age_sq_note = (
        f"  Negative quadratic → inverted-U; income peaks around age {vertex_age:.0f},\n"
        "  consistent with a classic Mincer earnings profile."
    )

educ_cols = sorted(
    [c for c in model.params.index if "education" in c],
    key=lambda c: int(c.split("[T.")[-1].rstrip("]")),
)
educ_lines = ""
for col in educ_cols:
    level = col.split("[T.")[-1].rstrip("]")
    pct   = (np.exp(model.params[col]) - 1) * 100
    pval  = model.pvalues[col]
    stars = "***" if pval < 0.001 else ("**" if pval < 0.01 else ("*" if pval < 0.05 else ""))
    educ_lines += f"  ISCED {level} vs ISCED 1 : {pct:+6.1f}%  {stars}\n"

print(f"""
╔══════════════════════════════════════════════════════════════════════╗
║  OLS: ln(equiv_income) ~ age + age² + education + sex + country     ║
║  N = {n_obs:,} working-age persons (25–64) across {len(COUNTRIES_REG)} countries          ║
╚══════════════════════════════════════════════════════════════════════╝

📐  MODEL FIT
  R²       = {r2:.3f}  → the model explains {r2*100:.1f}% of the variance in log-income.
             Low, but expected: income is also driven by sector, firm,
             household wealth, and luck — none of which we observe here.
  Adj. R²  = {adj_r2:.3f}  → almost identical to R²: with {n_obs:,} observations,
             {df_m} parameters cause negligible overfitting.
  F({df_m}, {n_obs - df_m - 1}) = {f_stat:,.1f},  p ≈ 0
             → the predictors jointly explain far more than chance.
  Resid. σ = {resid_s:.3f} log-pts → after accounting for demographics, individual
             incomes still vary by ±{resid_s:.2f} log-points (~{(np.exp(resid_s)-1)*100:.0f}% in €).

📅  AGE EFFECT  (centred on mean age ≈ {mean_age:.1f} years)
  Linear term  : {model.params['age_c']:+.4f} → each extra year of age is associated
                 with {age_pct:+.2f}% higher equivalised income, all else equal.
  Quadratic    : {age_sq_val:+.4f}
{age_sq_note}

🎓  EDUCATION PREMIUM vs ISCED 1  (primary / lower-secondary, reference)
  (*** p<0.001 | ** p<0.01 | * p<0.05)
{educ_lines}
  → A clear monotonic gradient: every ISCED step up brings a significant
    income gain. University graduates earn ~{(np.exp(model.params[educ_cols[-1]])-1)*100:.0f}% more than those
    with only primary education — a strong human-capital return.

💡  POLICY TAKEAWAY
  Education is the strongest demographic predictor of income within countries.
  Country fixed effects (not shown) absorb national wage levels, isolating
  the within-country education gradient — the parameter of interest for
  cross-national comparisons.
""")



╔══════════════════════════════════════════════════════════════════════╗
║  OLS: ln(equiv_income) ~ age + age² + education + sex + country     ║
║  N = 43,616 working-age persons (25–64) across 6 countries          ║
╚══════════════════════════════════════════════════════════════════════╝

📐  MODEL FIT
  R²       = 0.141  → the model explains 14.1% of the variance in log-income.
             Low, but expected: income is also driven by sector, firm,
             household wealth, and luck — none of which we observe here.
  Adj. R²  = 0.140  → almost identical to R²: with 43,616 observations,
             13 parameters cause negligible overfitting.
  F(13, 43602) = 548.8,  p ≈ 0
             → the predictors jointly explain far more than chance.
  Resid. σ = 0.901 log-pts → after accounting for demographics, individual
             incomes still vary by ±0.90 log-points (~146% in €).

📅  AGE EFFECT  (centred on mean age ≈ 44.3 years)
  Linear term  : +0.0089 → each extra year of age is 

---
## 3 · Machine Learning Pipeline

### 3.1 Problem: Classify At-Risk-of-Poverty

We predict **AROP status** (binary) from person-level features:
- Age, sex, education level (from P-file)
- Country (acts as a proxy for labour-market context)

### ⚠️ What is data leakage — and why does `Pipeline` prevent it?

`StandardScaler` computes mean and standard deviation from the data it sees.  
If you scale the **entire dataset** before cross-validation, the scaler has peeked at test-fold statistics — that's **data leakage**. Your CV error will look optimistically good, but the model will underperform on genuinely new data.

```
WITHOUT Pipeline (leaks test data into scaling):   WITH Pipeline (no leakage):

  all data                                            CV fold i
      │                                                  ├── train split
  StandardScaler.fit_transform(all)   ←──LEAK            │     └── pipeline.fit()
      │                                                  │           ├── scaler.fit_transform(train)
  train/test split                                       │           └── clf.fit(scaled_train)
      │                                                  └── test split
  cross_val_score  ← biased!                                    └── scaler.transform(test) ← no leak
```

A scikit-learn `Pipeline` bundles preprocessing + model into **one object**. When `cross_val_score` calls `.fit()` on each fold, the entire preprocessing chain runs only on that fold's training data.


In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, roc_auc_score,
    accuracy_score, balanced_accuracy_score,
)

# Compute AROP labels using country-specific thresholds
country_medians = df_reg.groupby("country")["equiv_income"].median()
df_ml = df_reg.copy()
df_ml["threshold"] = df_ml["country"].map(country_medians) * 0.60
df_ml["at_risk"]   = (df_ml["equiv_income"] < df_ml["threshold"]).astype(int)

print(f"ML dataset: {len(df_ml):,} persons")
print(f"At-risk rate: {df_ml['at_risk'].mean():.1%}  "
      f"(not-at-risk: {1 - df_ml['at_risk'].mean():.1%})")

# Encode country as integer
country_order = sorted(df_ml["country"].unique())
df_ml["country_enc"] = pd.Categorical(df_ml["country"], categories=country_order).codes

# ── Features  ────────────────────────
FEATURES = [
    "age", "sex", "education", "country_enc",
    "employment",                                 # PL031: 1=FT, 5=unemployed, 6=retired …
    "marital",                                    # PB190: 1=single, 2=married, 3=sep …
    "hh_size",                                    # HX040: number of persons in household
    "tenure",                                     # HH021: 1=owner, 3=market tenant …
    "warm_home",                                  # HH050: 1=can keep warm, 2=cannot
]

X = df_ml[FEATURES].copy()
y = df_ml["at_risk"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Train: {len(X_train):,}  Test: {len(X_test):,}")
print(f"Majority-class baseline accuracy: {max(y_test.mean(), 1-y_test.mean()):.1%}  "
      f"(always predict 'not at risk')")


ML dataset: 64,530 persons
At-risk rate: 25.2%  (not-at-risk: 74.8%)
Train: 51,624  Test: 12,906
Majority-class baseline accuracy: 74.8%  (always predict 'not at risk')


### 3.2 Pipeline Architecture

`ColumnTransformer` applies **different preprocessing to different feature groups** in one step.

```
Input features (9 columns)
│
├── numeric: ["age", "education", "hh_size"]
│     ├── SimpleImputer(strategy="median")    ← fill NaN with column median
│     └── StandardScaler()                   ← rescale to mean=0, std=1
│
└── categorical: ["sex", "country_enc",
                  "employment",  ← PL031: 1=FT emp … 5=unemployed … 6=retired
                  "marital",     ← PB190: 1=single, 2=married, 3=separated …
                  "tenure",      ← HH021: 1=owner, 2=mortgage, 3=market rent …
                  "warm_home"]   ← HH050: 1=can keep warm, 2=cannot
      └── SimpleImputer(strategy="most_frequent")  ← fill NaN with mode
            (integer-coded, no OneHotEncoder — RF splits on raw codes)
│
└── RandomForestClassifier(n_estimators=200, max_depth=6, class_weight="balanced")
      └── .predict_proba()  →  P(at_risk=1) for each person
```

> **Why `class_weight="balanced"`?**  
> At-risk households are ~25% of the data. Without weighting, the forest predicts "not at risk" for everyone and achieves 75% accuracy doing nothing useful.  
> `balanced` weights the minority class ×3, pushing recall on the at-risk group up from 0% to ~56%.


In [ ]:
# ── Numeric features: continuous / ordinal with meaningful scale ──────────────
numeric_features     = ["age", "education", "hh_size"]

# ── Categorical features: integer-coded but no ordinal meaning ────────────────
categorical_features = ["sex", "country_enc", "employment", "marital",
                        "tenure", "warm_home"]

preprocessor = ColumnTransformer(transformers=[
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale",  StandardScaler()),
    ]), numeric_features),
    ("cat", SimpleImputer(strategy="most_frequent"), categorical_features),
])

def make_rf_pipeline(class_weight=None):
    """Build a fresh pipeline; class_weight=None → uniform, 'balanced' → minority ×3."""
    return Pipeline([
        ("preprocessor", preprocessor),
        ("classifier",   RandomForestClassifier(
            n_estimators=200, max_depth=6, min_samples_leaf=20,
            class_weight=class_weight,
            random_state=42, n_jobs=-1,
        )),
    ])

def evaluate_pipeline(pipe, label):
    cv         = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    auc_cv     = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="roc_auc")
    pipe.fit(X_train, y_train)
    y_pred_  = pipe.predict(X_test)
    y_proba_ = pipe.predict_proba(X_test)[:, 1]

    acc  = accuracy_score(y_test, y_pred_)
    bacc = balanced_accuracy_score(y_test, y_pred_)
    auc  = roc_auc_score(y_test, y_proba_)

    print(f"{'─'*58}")
    print(f"  {label}")
    print(f"{'─'*58}")
    print(f"  CV AUC (5-fold)  : {auc_cv.mean():.4f} ± {auc_cv.std():.4f}")
    print(f"  Test AUC-ROC     : {auc:.4f}")
    print(f"  Accuracy         : {acc:.4f}")
    print(f"  Balanced Accuracy: {bacc:.4f}  (= avg recall across both classes)")
    print()
    print(classification_report(y_test, y_pred_, target_names=["Not at risk", "At risk"]))
    return pipe

# ── A: uniform weights (default) ─────────────────────────────────────────────
pipeline     = evaluate_pipeline(make_rf_pipeline(class_weight=None),
                                 "A — Uniform weights (default)")

# ── B: balanced weights — at-risk class gets ×3 weight ───────────────────────
pipeline_bal = evaluate_pipeline(make_rf_pipeline(class_weight="balanced"),
                                 "B — Balanced weights  (class_weight='balanced')")

# Use the balanced pipeline downstream (better recall on the minority class)
pipeline = pipeline_bal
auc_scores = cross_val_score(pipeline, X_train, y_train,
                             cv=StratifiedKFold(5, shuffle=True, random_state=42),
                             scoring="roc_auc")


──────────────────────────────────────────────────────────
  A — Uniform weights (default)
──────────────────────────────────────────────────────────
  CV AUC (5-fold)  : 0.6741 ± 0.0049
  Test AUC-ROC     : 0.6748
  Accuracy         : 0.7538
  Balanced Accuracy: 0.5177  (= avg recall across both classes)

              precision    recall  f1-score   support

 Not at risk       0.75      0.99      0.86      9656
     At risk       0.68      0.04      0.08      3250

    accuracy                           0.75     12906
   macro avg       0.72      0.52      0.47     12906
weighted avg       0.74      0.75      0.66     12906

──────────────────────────────────────────────────────────
  B — Balanced weights  (class_weight='balanced')
──────────────────────────────────────────────────────────
  CV AUC (5-fold)  : 0.6730 ± 0.0049
  Test AUC-ROC     : 0.6747
  Accuracy         : 0.6212
  Balanced Accuracy: 0.6277  (= avg recall across both classes)

              precision    recall  f1-s

### 3.3 Reading the Results

> 💡 **Why is AUC still "only" 0.67?**  
> The remaining gap to ~0.80 would require features not in the P/H-files:  
> *number of dependent children, work intensity of the household, housing-cost-to-income ratio.*  
> Variables like `HS120` (ability to make ends meet) or `HS140` (housing cost burden) **do** exist in the H-file and would push AUC further — but they measure financial strain directly, which overlaps with the AROP definition and risks being conceptual leakage.

---

#### The confusion matrix and the four cell types

|  | **Predicted: at risk** | **Predicted: not at risk** |
|--|--|--|
| **Actually at risk** | TP — true positive ✓ | FN — false negative (missed!) |
| **Actually not at risk** | FP — false positive (false alarm) | TN — true negative ✓ |

#### Precision and Recall

$$\text{Precision} = \frac{TP}{TP + FP}$$

*"Of all households we flagged as at-risk, what fraction actually is?"*  
→ FP hurts precision — false alarms dilute the flagged list.

$$\text{Recall} = \frac{TP}{TP + FN}$$

*"Of all truly at-risk households, what fraction did we catch?"*  
→ FN hurts recall — missed poor households drive this down.

$$\text{F1} = 2 \cdot \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$

*Harmonic mean — useful single number when both matter equally.*

#### The three summary metrics

| Metric | Formula | Good for | Trap |
|--------|---------|----------|------|
| **Accuracy** | $\frac{TP+TN}{TP+FP+FN+TN}$ | Balanced classes | Misleading with 75/25 split |
| **Balanced accuracy** | $\frac{1}{2}\!\left(\frac{TP}{TP+FN}+\frac{TN}{TN+FP}\right)$ | Imbalanced classes | — |
| **AUC-ROC** | area under ROC curve | Ranking / threshold-free | Doesn't reflect the prediction threshold |

> ⚠️ **The accuracy trap:** Model A (uniform weights) scores 75% accuracy — *identical to the majority-class baseline* — while recalling only 4% of at-risk persons. Always compare accuracy against the baseline.

#### Why `class_weight="balanced"` matters

`class_weight="balanced"` weights each at-risk sample by $\frac{n}{2 \cdot n_{\text{at risk}}} \approx \times 3$, making a missed at-risk person three times as costly to the loss function.

| Effect | Uniform | Balanced |
|--------|---------|----------|
| Recall "At risk" | 4% ← nearly useless | **64%** |
| Precision "At risk" | 68% | 36% |
| AUC-ROC | same (threshold-independent) | same |
| Balanced accuracy | 52% | **63%** |

**For poverty targeting**, recall is the priority — missing a vulnerable household is the costly mistake. 36% precision means roughly 2 false alarms per true positive, which may be acceptable when the intervention cost is low (e.g., mailing a benefit leaflet).


---
### 3.4 SHAP — Explaining Individual Predictions

**SHAP** (SHapley Additive exPlanations) answers the question: *"Why did the model predict this specific person is at risk?"*

It assigns each feature a **contribution score** for a given prediction, rooted in cooperative game theory (Shapley values). For each person:

- A **positive SHAP value** pushes the prediction *towards* at-risk
- A **negative SHAP value** pushes it *away* from at-risk
- The sum of all SHAP values + a baseline equals the model's output

| Output | What it shows |
|--------|--------------|
| **Global importance** | Mean \|SHAP\| across all persons — which features matter most overall |
| **Waterfall plot** | Feature-by-feature breakdown for one individual |

> `shap.TreeExplainer` is optimised for tree-based models (Random Forest, Gradient Boosting) and is much faster than the model-agnostic permutation approach.


In [ ]:
try:
    import shap

    rf_model = pipeline.named_steps["classifier"]
    X_test_t = pipeline.named_steps["preprocessor"].transform(X_test)
    feat_names = numeric_features + categorical_features

    explainer   = shap.TreeExplainer(rf_model)
    shap_values = explainer.shap_values(X_test_t)

    # For binary RF, shap_values is a list [class0, class1]
    sv = shap_values[1] if isinstance(shap_values, list) else shap_values[:, :, 1]

    # Global importance: mean |SHAP|
    mean_abs   = np.abs(sv).mean(axis=0)
    importance = pd.Series(mean_abs, index=feat_names).sort_values(ascending=False)

    print("Global SHAP feature importance:")
    for feat, val in importance.items():
        bar = "=" * int(val * 150)
        print(f"  {feat:<14}: {val:.4f}  {bar}")

    # Save waterfall for first at-risk person
    at_risk_idx = int(np.where(y_test.values == 1)[0][0])
    exp_val = (explainer.expected_value[1]
               if isinstance(explainer.expected_value, (list, np.ndarray))
               else explainer.expected_value)

    fig, ax = plt.subplots(figsize=(8, 4))
    shap.waterfall_plot(
        shap.Explanation(
            values=sv[at_risk_idx],
            base_values=float(exp_val),
            data=X_test_t[at_risk_idx],
            feature_names=feat_names,
        ),
        show=False,
    )
    plt.title("SHAP: Why is this person at-risk-of-poverty?")
    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / "shap_waterfall.png", dpi=100, bbox_inches="tight")
    plt.close()
    print("\nSHAP waterfall saved to shap_waterfall.png")

except ImportError:
    print("SHAP not installed — run: pip install shap")
except Exception as e:
    print(f"SHAP skipped ({type(e).__name__}): {e}")

Global SHAP feature importance:
  employment    : 0.0852  ============
  age           : 0.0497  =======
  hh_size       : 0.0299  ====
  education     : 0.0104  =
  sex           : 0.0060  
  country_enc   : 0.0055  
  marital       : 0.0028  
  tenure        : 0.0027  
  warm_home     : 0.0001  

SHAP waterfall saved to shap_waterfall.png


In [ ]:
%%writefile silc_toolkit/indicators.py
from __future__ import annotations
import math
import statistics
from typing import Optional, Sequence
import numpy as np


def median_income(incomes: Sequence[float]) -> float:
    pos = [x for x in incomes if x > 0]
    return statistics.median(pos) if pos else 0.0


def poverty_threshold(incomes: Sequence[float], threshold_pct: float = 0.60) -> float:
    return threshold_pct * median_income(incomes)


def arop_rate(
    incomes: Sequence[float],
    threshold_pct: float = 0.60,
    weights: Optional[Sequence[float]] = None,
) -> float:
    if not incomes:
        return 0.0
    thresh = poverty_threshold(incomes, threshold_pct)
    if weights is None:
        return sum(1 for i in incomes if i < thresh) / len(incomes)
    w_poor  = sum(w for i, w in zip(incomes, weights) if i < thresh)
    w_total = sum(weights)
    return w_poor / w_total if w_total > 0 else 0.0


def gini_coefficient(incomes: Sequence[float]) -> float:
    xs = sorted(x for x in incomes if x > 0)
    n  = len(xs)
    if n == 0:
        return 0.0
    total = sum(xs)
    if total == 0:
        return 0.0
    return sum(x * (2 * rank - n - 1) for rank, x in enumerate(xs, 1)) / (n * total)


def material_deprivation_rate(flags: Sequence[int], threshold: int = 3) -> float:
    if not flags:
        return 0.0
    return sum(1 for f in flags if f >= threshold) / len(flags)


def s80s20_ratio(incomes: Sequence[float]) -> float:
    xs = sorted(x for x in incomes if x > 0)
    n  = len(xs)
    if n < 5:
        return float("nan")
    q = n // 5
    return sum(xs[-q:]) / sum(xs[:q]) if sum(xs[:q]) > 0 else float("inf")


def arope_rate(
    at_risk: Sequence[bool],
    deprived: Sequence[bool],
    low_work: Sequence[bool],
) -> float:
    n = len(at_risk)
    if n == 0:
        return 0.0
    return sum(1 for a, d, w in zip(at_risk, deprived, low_work) if a or d or w) / n


def gini_numpy(incomes: np.ndarray) -> float:
    xs = np.sort(incomes[incomes > 0])
    n  = len(xs)
    if n == 0:
        return 0.0
    ranks   = np.arange(1, n + 1)
    weights = 2 * ranks - n - 1
    return float(np.dot(weights, xs) / (n * xs.sum()))


def gini_weighted(incomes: np.ndarray, weights: np.ndarray) -> float:
    mask   = incomes > 0
    xs, ws = incomes[mask], weights[mask]
    if xs.size == 0:
        return 0.0
    order   = np.argsort(xs)
    xs, ws  = xs[order], ws[order]
    cumw    = np.cumsum(ws)
    lorenz_y = np.cumsum(ws * xs) / np.dot(ws, xs)
    lorenz_x = cumw / cumw[-1]
    return float(1 - 2 * np.trapz(lorenz_y, lorenz_x))


def equivalised_income_numpy(
    disposable_income: np.ndarray,
    n_adults: np.ndarray,
    n_children: np.ndarray,
) -> np.ndarray:
    scale = 1.0 + 0.5 * (n_adults - 1) + 0.3 * n_children
    return disposable_income / np.maximum(scale, 1.0)

Overwriting silc_toolkit/indicators.py


---
### 🏋️ Exercise 1 — Weighted AROP

Design weights (`DB090` from the D-file) correct for unequal sampling probabilities.
Unweighted AROP is biased when some household types are over-represented.

1. Load H-file + D-file for Luxembourg 2012 using `load_household_df`
2. Compute **unweighted** and **weighted** AROP
3. How large is the difference in percentage points?

In [ ]:
# 🏋️ Exercise 1 — Your solution here!
from silc_toolkit.loaders import load_household_df

df_lu = load_household_df("LU", 2012, DATA_DIR)
print("Columns:", df_lu.columns.tolist())

# TODO: extract equiv_income and DB090 (weight), then call arop_rate with weights


Columns: ['HB010', 'HB020', 'HB030', 'HY010', 'HY020', 'HX040', 'DB090', 'DB040', 'equiv_income']


<details><summary>💡 Click for the solution</summary>

```python
from silc_toolkit.loaders import load_household_df
from silc_toolkit.indicators import arop_rate

df_lu  = load_household_df("LU", 2012, DATA_DIR)
df_pos = df_lu[df_lu["equiv_income"] > 0].copy()

incomes = df_pos["equiv_income"].tolist()
weights = df_pos["DB090"].fillna(1.0).tolist() if "DB090" in df_pos.columns else None

arop_uw = arop_rate(incomes)
arop_w  = arop_rate(incomes, weights=weights) if weights else None

print(f"Luxembourg 2012:")
print(f"  Unweighted AROP: {arop_uw:.3%}")
if arop_w:
    print(f"  Weighted AROP  : {arop_w:.3%}")
    print(f"  Difference     : {abs(arop_w - arop_uw)*100:.2f} pp")
```
</details>

---
### 🏋️ Exercise 2 — Improve the Classifier

1. Switch to `GradientBoostingClassifier` and compare AUC
2. Add `household_size` as a feature (from the Parquet files)
3. **Bonus:** Use `GridSearchCV` to tune `max_depth` over [3, 5, 7]

In [ ]:
# 🏋️ Exercise 2 — Improve the Classifier
# Step 1: replace RandomForestClassifier with GradientBoostingClassifier
#         (same Pipeline structure — that's the power of the sklearn API!)
from sklearn.ensemble import GradientBoostingClassifier

# your turn

<details><summary>💡 Click for the solution</summary>

```python
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, GridSearchCV

# Step 1 & 2 — GradientBoostingClassifier vs RandomForest
pipeline_gb = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier",   GradientBoostingClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42
    )),
])

auc_gb = cross_val_score(pipeline_gb, X_train, y_train, cv=cv, scoring="roc_auc").mean()
print(f"RF  AUC : {auc_scores.mean():.4f}")
print(f"GB  AUC : {auc_gb:.4f}")

# Bonus — tune max_depth with GridSearchCV
param_grid = {"classifier__max_depth": [3, 5, 7]}
grid = GridSearchCV(
    Pipeline([
        ("preprocessor", preprocessor),
        ("classifier",   GradientBoostingClassifier(
            n_estimators=200, learning_rate=0.05, random_state=42
        )),
    ]),
    param_grid, cv=cv, scoring="roc_auc", n_jobs=-1,
)
grid.fit(X_train, y_train)
print(f"Best max_depth : {grid.best_params_['classifier__max_depth']}")
print(f"Best CV AUC    : {grid.best_score_:.4f}")
```
</details>

---
## 🗺️ Recap

| Concept | Key syntax | SILC application |
|---------|-----------|------------------|
| **NumPy vectorise** | `np.sort()`, `np.dot()`, `np.arange()` | 100× faster Gini |
| **Weighted Gini** | `np.cumsum(ws*xs)` + `np.trapz()` | Correct for survey design weights |
| **statsmodels OLS** | `smf.ols(formula, data).fit()` | Income ~ education + age + country |
| **sklearn Pipeline** | `Pipeline([("step", obj)])` | No data leakage across CV folds |
| **ColumnTransformer** | numeric + categorical branches | Impute + scale + encode in one step |
| **Cross-validation** | `cross_val_score(pipeline, X, y, cv=5)` | Unbiased AUC estimate |
| **SHAP** | `shap.TreeExplainer(rf)` | Explain poverty-risk predictions |

---
## ⏭️ Up Next

**Notebook 06 — MCP Servers & Interactive Visualisation** (this afternoon)

- Wrap AROP and Gini as MCP tools — Claude Desktop answers your statistics questions
- Plotly animated income distribution by country and year
- Geopandas choropleth: regional poverty across the EU

☕ *Lunch break — back at 13:30!*